In [1]:
import pandas as pd

# Загрузка данных
df = pd.read_csv('maudau.csv')

# Приведение колонки date к формату datetime
df['date'] = pd.to_datetime(df['date'])

# Фильтрация данных за ноябрь 2023 года
nov_df = df[(df['date'].dt.year == 2023) & (df['date'].dt.month == 11)]

# Расчёт MAU (количество уникальных пользователей за ноябрь)
mau_november = nov_df['user_id'].nunique()

print(f"MAU за ноябрь: {mau_november}")

MAU за ноябрь: 7639


In [2]:
import pandas as pd

df = pd.read_csv('maudau.csv')
df['date'] = pd.to_datetime(df['date'])

# 1. DAU по каждому дню за весь период
dau_daily = df.groupby('date')['user_id'].nunique().reset_index()
dau_daily.columns = ['date', 'dau']

# 2. Средний общий DAU (показатель одной цифрой)
avg_dau = dau_daily['dau'].mean()
print(f"\nСредний DAU за весь период: {avg_dau:.2f}")


Средний DAU за весь период: 560.47


In [3]:
# 1. Находим дату первого визита (когорту) каждого пользователя
first_visits = df.groupby('user_id')['date'].min().reset_index()
first_visits.columns = ['user_id', 'cohort_date']

# 2. Фильтруем пользователей когорты 1 ноября
cohort_nov_1 = first_visits[first_visits['cohort_date'] == '2023-11-01']
cohort_size = len(cohort_nov_1)

# 3. Проверяем, кто из этой когорты зашел на 1-й день (2 ноября)
day_1_date = pd.to_datetime('2023-11-02')
returned_day_1 = df[
    (df['user_id'].isin(cohort_nov_1['user_id'])) & 
    (df['date'] == day_1_date)
]['user_id'].nunique()

# 4. Считаем Retention Day 1
retention_day_1 = (returned_day_1 / cohort_size) * 100 if cohort_size > 0 else 0

print(f"Размер когорты (01.11): {cohort_size}")
print(f"Вернулись на 1-й день (02.11): {returned_day_1}")
print(f"Retention Day 1: {retention_day_1:.2f}%")

Размер когорты (01.11): 623
Вернулись на 1-й день (02.11): 166
Retention Day 1: 26.65%


In [4]:
# 1. Общее количество уникальных пользователей за ноябрь (MAU)
total_users = nov_df['user_id'].nunique()

# 2. Количество уникальных пользователей, посмотревших хотя бы 1 объявление (view_adverts > 0)
users_with_views = nov_df[nov_df['view_adverts'] > 0]['user_id'].nunique()

# 3. Расчёт пользовательской конверсии
user_conversion = (users_with_views / total_users) * 100

print(f"Всего пользователей за ноябрь: {total_users}")
print(f"Пользователей с просмотрами (>0): {users_with_views}")
print(f"Пользовательская конверсия: {user_conversion:.2f}%")

Всего пользователей за ноябрь: 7639
Пользователей с просмотрами (>0): 3538
Пользовательская конверсия: 46.31%


In [5]:
nov_df = df[(df['date'].dt.year == 2023) & (df['date'].dt.month == 11)]

# Вариант 1: Через сумму просмотров всех пользователей / MAU
total_views = nov_df['view_adverts'].sum()
unique_users = nov_df['user_id'].nunique()

avg_views_per_user = total_views / unique_users

# Вариант 2: Сначала суммируем просмотры каждого пользователя, затем берем среднее
user_total_views = nov_df.groupby('user_id')['view_adverts'].sum()
avg_views_per_user_v2 = user_total_views.mean()

print(f"Среднее количество просмотров на пользователя: {avg_views_per_user:.2f}")

Среднее количество просмотров на пользователя: 2.87


In [6]:
total_users = 2000
promoters = 1200  # Сторонники
detractors = 500  # Критики
passives = 300    # Нейтралы

# Расчёт процентных долей
pct_promoters = (promoters / total_users) * 100
pct_detractors = (detractors / total_users) * 100

# Расчёт NPS
nps = pct_promoters - pct_detractors

print(f"Доля сторонников: {pct_promoters:.1f}%")
print(f"Доля критиков: {pct_detractors:.1f}%")
print(f"NPS: {nps:.1f}%")

Доля сторонников: 60.0%
Доля критиков: 25.0%
NPS: 35.0%


In [10]:
import pandas as pd
from scipy import stats

# 1. Загрузка данных
df = pd.read_csv('Данные АБ-тестов.csv')

# 2. Анализ каждого эксперимента
for exp_id in sorted(df['experiment_num'].unique()):
    exp_data = df[df['experiment_num'] == exp_id]
    
    control = exp_data[exp_data['experiment_group'] == 'control']['revenue']
    test = exp_data[exp_data['experiment_group'] == 'test']['revenue']
    
    arpu_control = control.mean()
    arpu_test = test.mean()
    
    # Сравнение распределений (t-test / Welch's t-test)
    t_stat, p_val_ttest = stats.ttest_ind(test, control, equal_var=False)
    
    # Непарометрический критерий Манна-Уитни (полезен при скошенном распределении revenue)
    stat_u, p_val_mw = stats.mannwhitneyu(test, control, alternative='two-sided')
    
    print(f"=== ЭКСПЕРИМЕНТ №{exp_id} ===")
    print(f"Размер Control: {len(control)}, ARPU Control: {arpu_control:.2f}")
    print(f"Размер Test:    {len(test)}, ARPU Test:    {arpu_test:.2f}")
    print(f"Δ ARPU: {arpu_test - arpu_control:.2f} ({((arpu_test - arpu_control) / arpu_control)*100:.2f}%)")
    print(f"p-value (Welch's t-test): {p_val_ttest:.5f}")
    print(f"p-value (Mann-Whitney U): {p_val_mw:.5f}")
    print("-" * 40)

=== ЭКСПЕРИМЕНТ №1 ===
Размер Control: 465, ARPU Control: 722.46
Размер Test:    480, ARPU Test:    665.74
Δ ARPU: -56.72 (-7.85%)
p-value (Welch's t-test): 0.68897
p-value (Mann-Whitney U): 0.79601
----------------------------------------
=== ЭКСПЕРИМЕНТ №2 ===
Размер Control: 465, ARPU Control: 704.65
Размер Test:    480, ARPU Test:    332.93
Δ ARPU: -371.72 (-52.75%)
p-value (Welch's t-test): 0.00113
p-value (Mann-Whitney U): 0.00845
----------------------------------------
=== ЭКСПЕРИМЕНТ №3 ===
Размер Control: 465, ARPU Control: 663.21
Размер Test:    480, ARPU Test:    998.67
Δ ARPU: 335.46 (50.58%)
p-value (Welch's t-test): 0.06032
p-value (Mann-Whitney U): 0.00100
----------------------------------------


In [8]:
df = pd.read_csv('листеры.csv')

# 1. Считаем суммарный доход по каждому пользователю
user_revenue = df.groupby('user_id')['revenue'].sum()

# 2. Считаем среднее значение дохода на пользователя
arpu = user_revenue.mean()

print(f"Средний доход на пользователя (ARPU): {arpu:.2f}")

Средний доход на пользователя (ARPU): 156.48


In [9]:
median_age = df['age'].median()
print(f"Медиана возраста по всем строкам: {median_age}")

Медиана возраста по всем строкам: 28.0
